In [1]:
import pandas as pd
import numpy as np
import joblib
import folium
import os

In [2]:
model = joblib.load(
    "../models/area_risk_model.joblib"
)

features = joblib.load(
    "../models/area_risk_features.joblib"
)

thresholds = joblib.load(
    "../models/area_risk_thresholds.joblib"
)

print("Model loaded ✅")
print("Features:", len(features))
print("Thresholds:", thresholds)

Model loaded ✅
Features: 18
Thresholds: {'low_medium': 4.4105490445865145, 'medium_high': 10.652076331270639}


In [4]:
history = pd.read_csv(
    "../data/grid_week_history.csv"
)

history['week'] = pd.to_datetime(
    history['week']
)

history = history.sort_values(
    ['grid_id', 'week']
).reset_index(drop=True)

print(history.shape)
print(history['week'].min())
print(history['week'].max())

(220934, 21)
2020-12-28 00:00:00
2026-09-28 00:00:00


In [5]:
last_week = history['week'].max()

next_week = (
    last_week
    + pd.Timedelta(days=7)
)

print("Last observed week:", last_week)
print("Forecast week:", next_week)

Last observed week: 2026-09-28 00:00:00
Forecast week: 2026-10-05 00:00:00


In [6]:
grid_info = (
    history
    .groupby('grid_id')
    [['grid_lat', 'grid_lon']]
    .first()
    .reset_index()
)

future = grid_info.copy()

future['week'] = next_week

print("Areas to forecast:", len(future))
future.head()

Areas to forecast: 734


,grid_id,grid_lat,grid_lon,week
0,36.62_-91.69,36.62,-91.69,2026-10-05
1,41.64_-87.54,41.64,-87.54,2026-10-05
2,41.64_-87.6,41.64,-87.60,2026-10-05
3,41.64_-87.61,41.64,-87.61,2026-10-05
4,41.64_-87.62,41.64,-87.62,2026-10-05


In [7]:
future['week_of_year'] = (
    next_week.isocalendar().week
)

future['month'] = next_week.month
future['year'] = next_week.year

future['week_sin'] = np.sin(
    2 * np.pi
    * future['week_of_year']
    / 52
)

future['week_cos'] = np.cos(
    2 * np.pi
    * future['week_of_year']
    / 52
)

future['month_sin'] = np.sin(
    2 * np.pi
    * (future['month'] - 1)
    / 12
)

future['month_cos'] = np.cos(
    2 * np.pi
    * (future['month'] - 1)
    / 12
)

In [8]:
def get_lag(group, lag):

    group = group.sort_values('week')

    if len(group) < lag:
        return np.nan

    return group.iloc[-lag]['crime_count']

In [9]:
history_groups = {
    grid_id: group.sort_values('week')
    for grid_id, group
    in history.groupby('grid_id')
}

for lag in [1, 2, 4, 8, 12]:

    future[f'lag_{lag}'] = (
        future['grid_id']
        .map(
            lambda grid_id:
                get_lag(
                    history_groups[grid_id],
                    lag
                )
        )
    )

In [11]:
def historical_stats(grid_id):

    values = (
        history_groups[grid_id]
        ['crime_count']
    )

    return pd.Series({
        'rolling_4_mean':
            values.tail(4).mean(),

        'rolling_8_mean':
            values.tail(8).mean(),

        'rolling_4_std':
            values.tail(4).std(),

        'historical_mean':
            values.mean()
    })


In [12]:
stats = (
    future['grid_id']
    .apply(historical_stats)
)

future[
    [
        'rolling_4_mean',
        'rolling_8_mean',
        'rolling_4_std',
        'historical_mean'
    ]
] = stats.values

In [13]:
print(
    future[features]
    .isnull()
    .sum()
)

future[
    [
        'grid_id',
        'lag_1',
        'lag_4',
        'rolling_4_mean',
        'historical_mean'
    ]
].head()

grid_lat           0
grid_lon           0
week_of_year       0
month              0
year               0
week_sin           0
week_cos           0
month_sin          0
month_cos          0
lag_1              0
lag_2              0
lag_4              0
lag_8              0
lag_12             0
rolling_4_mean     0
rolling_8_mean     0
rolling_4_std      0
historical_mean    0
dtype: int64


,grid_id,lag_1,lag_4,rolling_4_mean,historical_mean
0,36.62_-91.69,0,0,0.0,0.006645
1,41.64_-87.54,0,0,0.0,0.019934
2,41.64_-87.6,0,0,0.0,0.036545
3,41.64_-87.61,0,0,0.0,0.056478
4,41.64_-87.62,0,0,0.0,0.046512


In [14]:
X_future = future[features]

future['predicted_crimes'] = (
    model.predict(X_future)
)

future['predicted_crimes'] = (
    future['predicted_crimes']
    .clip(lower=0)
)

future['predicted_crimes'].describe()

count    734.000000
mean       4.985682
std        5.613464
min        0.010401
25%        0.910035
50%        3.477537
75%        7.192614
max       60.836991
Name: predicted_crimes, dtype: float64

In [15]:
low_threshold = thresholds['low_medium']
high_threshold = thresholds['medium_high']


def assign_risk(value):

    if value >= high_threshold:
        return 'HIGH'

    elif value >= low_threshold:
        return 'MEDIUM'

    return 'LOW'


future['risk_level'] = (
    future['predicted_crimes']
    .apply(assign_risk)
)

In [16]:
future = (
    future
    .sort_values(
        'predicted_crimes',
        ascending=False
    )
    .reset_index(drop=True)
)

future['hotspot_rank'] = (
    np.arange(
        1,
        len(future) + 1
    )
)

In [17]:
top_hotspots = future[
    [
        'hotspot_rank',
        'grid_id',
        'grid_lat',
        'grid_lon',
        'predicted_crimes',
        'risk_level'
    ]
].head(20)

top_hotspots

,hotspot_rank,grid_id,grid_lat,grid_lon,predicted_crimes,risk_level
0,1,41.89_-87.63,41.89,-87.63,60.836991,HIGH
1,2,41.88_-87.63,41.88,-87.63,55.220352,HIGH
2,3,41.9_-87.63,41.90,-87.63,32.411434,HIGH
3,4,41.89_-87.62,41.89,-87.62,32.239775,HIGH
4,5,41.76_-87.56,41.76,-87.56,22.550982,HIGH
5,6,41.87_-87.63,41.87,-87.63,22.413752,HIGH
6,7,41.77_-87.58,41.77,-87.58,21.958858,HIGH
7,8,41.88_-87.73,41.88,-87.73,21.654326,HIGH
8,9,41.75_-87.61,41.75,-87.61,20.869654,HIGH
9,10,41.75_-87.6,41.75,-87.60,20.333620,HIGH


In [18]:
hotspot_map = folium.Map(
    location=[
        future['grid_lat'].mean(),
        future['grid_lon'].mean()
    ],
    zoom_start=10,
    tiles='OpenStreetMap'
)

colors = {
    'LOW': 'green',
    'MEDIUM': 'orange',
    'HIGH': 'red'
}

In [19]:
for _, row in future.iterrows():

    color = colors[
        row['risk_level']
    ]

    folium.CircleMarker(
        location=[
            row['grid_lat'],
            row['grid_lon']
        ],

        radius=6,

        color=color,

        fill=True,

        fill_color=color,

        fill_opacity=0.65,

        popup=folium.Popup(
            f"""
            <b>Future Hotspot Rank:</b>
            {int(row['hotspot_rank'])}<br>

            <b>Grid:</b>
            {row['grid_id']}<br>

            <b>Forecast Week:</b>
            {row['week'].date()}<br>

            <b>Predicted Crimes:</b>
            {row['predicted_crimes']:.2f}<br>

            <b>Risk:</b>
            {row['risk_level']}
            """,
            max_width=280
        )

    ).add_to(hotspot_map)

In [20]:
hotspot_map

In [21]:
future.to_csv(
    "../data/future_hotspots.csv",
    index=False
)

top_hotspots.to_csv(
    "../data/top_future_hotspots.csv",
    index=False
)

hotspot_map.save(
    "future_hotspots_map.html"
)

print("Future hotspot outputs saved ✅")

Future hotspot outputs saved ✅


In [22]:
outputs = [
    "../data/future_hotspots.csv",
    "../data/top_future_hotspots.csv",
    "future_hotspots_map.html"
]

for path in outputs:

    print(
        path,
        "→",
        os.path.exists(path)
    )

../data/future_hotspots.csv → True
../data/top_future_hotspots.csv → True
future_hotspots_map.html → True
